<a href="https://colab.research.google.com/github/Adhiaris/Practical-Statistics-for-Data-Scientist-Books/blob/main/Practical_Statistics_Chapter5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 5: Classification

Classification predicts which category a record belongs to. Often we want the **predicted probability** (propensity) for the class of interest, then apply a cutoff to make a decision.

This chapter covers: **Naive Bayes**, **Discriminant Analysis**, **Logistic Regression**, **Model Evaluation** (confusion matrix, ROC, AUC), and **Imbalanced Data** strategies.


## Setup
Import required Python packages.


In [ ]:
!pip install -q pygam dmba

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.naive_bayes import MultinomialNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.metrics import roc_curve, accuracy_score, roc_auc_score

import statsmodels.api as sm

from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from pygam import LinearGAM, s, f, l


from dmba import classificationSummary

import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

We load scikit-learn classifiers, evaluation metrics, statsmodels for detailed GLM output, imbalanced-learn for oversampling, pygam for additive models, and the dmba companion package.


In [ ]:
try:
    import common
    DATA = common.dataDirectory()
except ImportError:
    DATA = Path().resolve() / 'data'

Data path uses the textbook's `common` module if available, otherwise defaults to a local `data/` directory.


Set paths to the three datasets used in this chapter.


In [ ]:
!git clone https://github.com/gedeck/practical-statistics-for-data-scientists.git
DATA = Path('practical-statistics-for-data-scientists/data')

LOAN3000_CSV = DATA / 'loan3000.csv'
LOAN_DATA_CSV = DATA / 'loan_data.csv.gz'
FULL_TRAIN_SET_CSV = DATA / 'full_train_set.csv.gz'

Three datasets:
- `loan_data.csv.gz` (45,342 records) — main dataset with loan outcomes.
- `loan3000.csv` (3,000 records) — smaller subset for visualization.
- `full_train_set.csv.gz` (119,987 records) — full set with realistic class imbalance.


# Naive Bayes
## The Naive Solution


**Naive Bayes** applies Bayes' theorem assuming predictors are conditionally independent given the class:
$$P(Y=i \mid X_1,\ldots,X_p) = \frac{P(Y=i)\prod_j P(X_j \mid Y=i)}{\sum_k P(Y=k)\prod_j P(X_j \mid Y=k)}$$

Despite the unrealistic independence assumption, naive Bayes often produces reliable class **rankings**.


In [ ]:
loan_data = pd.read_csv(LOAN_DATA_CSV)

loan_data.outcome = loan_data.outcome.astype('category')
loan_data.outcome.cat.reorder_categories(['paid off', 'default'])
loan_data.purpose_ = loan_data.purpose_.astype('category')
loan_data.home_ = loan_data.home_.astype('category')
loan_data.emp_len_ = loan_data.emp_len_.astype('category')

predictors = ['purpose_', 'home_', 'emp_len_']
outcome = 'outcome'
X = pd.get_dummies(loan_data[predictors], prefix='', prefix_sep='', dtype=int)
y = loan_data[outcome]

naive_model = MultinomialNB(alpha=0.01, fit_prior=True)
naive_model = MultinomialNB(alpha=1e-10, fit_prior=False)
naive_model.fit(X, y)

new_loan = X.loc[146:146, :]
print('predicted class: ', naive_model.predict(new_loan)[0])

probabilities = pd.DataFrame(naive_model.predict_proba(new_loan),
                             columns=naive_model.classes_)
print('predicted probabilities',)
print(probabilities)

**Preparation:** Categorical predictors are one-hot encoded. `MultinomialNB` requires non-negative features.

**Result for record 146:** $P(\text{default})=0.654$ — the model predicts default. Note that naive Bayes probability estimates are often **poorly calibrated**, but rankings are generally reliable.


### Example not in book


Continuous variables must be binned (`MultinomialNB`) or modeled with `GaussianNB`. Laplace smoothing (`alpha`) prevents zero probabilities when a category is absent from one class in training.


# Discriminant Analysis
## A Simple Example


**LDA** (Fisher, 1936) finds a linear combination that maximizes class separation relative to within-class spread:
$$\frac{SS_{\text{between}}}{SS_{\text{within}}}$$

The covariance matrix $\boldsymbol{\Sigma}$ accounts for variable correlations in the within-class scatter.


In [ ]:
loan3000 = pd.read_csv(LOAN3000_CSV)
loan3000.outcome = loan3000.outcome.astype('category')

predictors = ['borrower_score', 'payment_inc_ratio']
outcome = 'outcome'

X = loan3000[predictors]
y = loan3000[outcome]

loan_lda = LinearDiscriminantAnalysis()
loan_lda.fit(X, y)
print(pd.DataFrame(loan_lda.scalings_, index=X.columns))

The discriminant weights define:
$$\text{LD1} = 7.176 \times \texttt{borrower\_score} - 0.100 \times \texttt{payment\_inc\_ratio}$$

`borrower_score` dominates (~72× larger weight), confirming it is the primary separator. Higher scores → paid off; higher payment ratios → default.


In [ ]:
pred = pd.DataFrame(loan_lda.predict_proba(loan3000[predictors]),
                    columns=loan_lda.classes_)
print(pred.head())

LDA posterior probabilities: records near 0.5 sit close to the decision boundary; records near 0 or 1 are classified with high confidence.


### Figure 5-1


In [ ]:
center = np.mean(loan_lda.means_, axis=0)
slope = - loan_lda.scalings_[0] / loan_lda.scalings_[1]
intercept = center[1] - center[0] * slope

x_0 = (0 - intercept) / slope
x_20 = (20 - intercept) / slope

lda_df = pd.concat([loan3000, pred['default']], axis=1)
lda_df.head()

fig, ax = plt.subplots(figsize=(4, 4))
g = sns.scatterplot(x='borrower_score', y='payment_inc_ratio',
                    hue='default', data=lda_df,
                    palette=sns.diverging_palette(240, 10, n=9, as_cmap=True),
                    ax=ax, legend=False)

ax.set_ylim(0, 20)
ax.set_xlim(0.15, 0.8)
ax.plot((x_0, x_20), (0, 20), linewidth=3)
ax.plot(*loan_lda.means_.transpose())

plt.tight_layout()
plt.show()

**Figure 5-1:** The diagonal line is the decision boundary where both classes are equally likely. Its steep slope confirms `borrower_score` dominates. Upper-left (low score, high ratio) → default; lower-right → paid off. LDA always produces a **linear** boundary.


# Logistic Regression
## Logistic Response Function and Logit


**Logistic regression** links a linear predictor to a probability via three transformations:

**Logistic function:** $p = \dfrac{1}{1+e^{-(\beta_0+\beta_1 x_1+\cdots)}}$

**Odds:** $\dfrac{p}{1-p}$

**Logit (log-odds):** $\log\!\left(\dfrac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \cdots$


In [ ]:
p = np.arange(0.01, 1, 0.01)
df = pd.DataFrame({
    'p': p,
    'logit': np.log(p / (1 - p)),
    'odds': p / (1 - p),
})

fig, ax = plt.subplots(figsize=(3, 3))
ax.axhline(0, color='grey', linestyle='--')
ax.axvline(0.5, color='grey', linestyle='--')
ax.plot(df['p'], df['logit'])
ax.set_xlabel('Probability')
ax.set_ylabel('logit(p)')

plt.tight_layout()
plt.show()

**Figure 5-2:** The logit maps $(0,1) \to (-\infty,+\infty)$, crossing zero at $p=0.5$. This makes probabilities a valid target for a linear model; the sigmoid reverses the mapping to recover probabilities from predictions.


## Logistic Regression and the GLM
`LogisticRegression` in scikit-learn fits a logit model; statsmodels' GLM with binomial family gives detailed statistical output (standard errors, z-values, p-values).


In [ ]:
predictors = ['payment_inc_ratio', 'purpose_', 'home_', 'emp_len_',
              'borrower_score']
outcome = 'outcome'
X = pd.get_dummies(loan_data[predictors], prefix='', prefix_sep='',
                   drop_first=True, dtype=int)
y = loan_data[outcome]

logit_reg = LogisticRegression(penalty='l2', C=1e42, solver='liblinear')
logit_reg.fit(X, y)

print('intercept ', logit_reg.intercept_[0])
print('classes', logit_reg.classes_)
pd.DataFrame({'coeff': logit_reg.coef_[0]},
             index=X.columns)

`C=1e42` disables regularization to match R's `glm()`. `drop_first=True` avoids the dummy variable trap; the dropped category becomes the reference level.


Note: intercept and coefficient signs are reversed vs. the R model due to class ordering convention.


Scikit-learn orders classes alphabetically, so `coef_` predicts $P(\text{paid off})$. Signs are therefore opposite to R's convention, which models $P(\text{default})$.


In [ ]:
print(loan_data['purpose_'].cat.categories)
print(loan_data['home_'].cat.categories)
print(loan_data['emp_len_'].cat.categories)

Reference levels: `credit_card` (purpose), `MORTGAGE` (home), `< 1 Year` (employment). This produces 11 predictor columns total.


*Not in book:* Use `OrdinalEncoder` to manually assign numeric codes to ordinal or outcome variables.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder(categories=[['paid off', 'default']])
y_enc = enc.fit_transform(loan_data[[outcome]]).ravel()

logit_reg_enc = LogisticRegression(penalty="l2", C=1e42, solver='liblinear')
logit_reg_enc.fit(X, y_enc)

print('intercept ', logit_reg_enc.intercept_[0])
print('classes', logit_reg_enc.classes_)
pd.DataFrame({'coeff': logit_reg_enc.coef_[0]},
             index=X.columns)

Encoding `paid off`→0, `default`→1 flips all signs — now positive coefficients mean higher default risk, matching R convention. The model's predictive behavior is identical.


## Predicted Values from Logistic Regression


In [ ]:
pred = pd.DataFrame(logit_reg.predict_log_proba(X),
                    columns=logit_reg.classes_)
print(pred.describe())

`predict_log_proba()` returns log probabilities (always negative). Log scale avoids numerical underflow when multiplying many small probabilities.


In [ ]:
pred = pd.DataFrame(logit_reg.predict_proba(X),
                    columns=logit_reg.classes_)
print(pred.describe())

`predict_proba()` gives raw probabilities in $[0,1]$. The mean is exactly 0.5 — a mathematical property of unpenalized logistic regression on balanced data. Predicted default probabilities range from 0.063 to 0.971.


## Interpreting the Coefficients and Odds Ratios


In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax.plot(df['logit'], df['odds'])
ax.set_xlabel('log(odds ratio)')
ax.set_ylabel('odds ratio')
ax.set_xlim(0, 5.1)
ax.set_ylim(-5, 105)

plt.tight_layout()
plt.show()

**Figure 5-3:** Shows the exponential relationship between log-odds ($\beta_j$) and odds ratio ($e^{\beta_j}$). Small log-odds differences become large odds-ratio differences at higher values. Example: `small_business` coefficient → $e^{1.22} \approx 3.4\times$ default odds vs. credit card loans.


## Assessing the Model
For comparison, statsmodels GLM requires a numeric (0/1) outcome.


In [ ]:
y_numbers = [1 if yi == 'default' else 0 for yi in y]
logit_reg_sm = sm.GLM(y_numbers, X.assign(const=1),
                      family=sm.families.Binomial())
logit_result = logit_reg_sm.fit()
print(logit_result.summary())

Key statsmodels output columns: `coef` (log-odds), `std err` (precision), `z` (significance), `P>|z|` (p-value), `[0.025, 0.975]` (95% CI).

Notable odds ratios: `small_business` $e^{1.22}\approx3.4\times$ default odds; `borrower_score` $e^{-4.61}\approx0.01$ (100× safer at top vs. bottom). OWN (`p=0.204`) is not significant. AUC = 0.69 is a modest but meaningful result with only five predictors.


Use splines


In [ ]:
import statsmodels.formula.api as smf
formula = ('outcome ~ bs(payment_inc_ratio, df=8) + purpose_ + ' +
           'home_ + emp_len_ + bs(borrower_score, df=3)')
model = smf.glm(formula=formula, data=loan_data, family=sm.families.Binomial())
results = model.fit()
print(results.summary())

**B-splines** (`bs(x, df=n)`) allow nonlinear predictor effects within the GLM. The spline coefficients define the curve shape and aren't directly interpretable — visualize them with partial residual plots instead.


In [ ]:
from statsmodels.genmod.generalized_linear_model import GLMResults
def partialResidualPlot(model, df, outcome, feature, fig, ax):
    y_actual = [0 if s == 'default' else 1 for s in df[outcome]]
    y_pred = model.predict(df)
    org_params = model.params.copy()
    zero_params = model.params.copy()
    for i, name in enumerate(zero_params.index):
        if feature in name:
            continue
        zero_params.iloc[i] = 0.0
    model.initialize(model.model, zero_params)
    feature_prediction = model.predict(df)
    ypartial = -np.log(1/feature_prediction - 1)
    ypartial = ypartial - np.mean(ypartial)
    model.initialize(model.model, org_params)
    results = pd.DataFrame({
        'feature': df[feature],
        'residual': -2 * (y_actual - y_pred),
        'ypartial': ypartial/ 2,
    })
    results = results.sort_values(by=['feature'])

    ax.scatter(results.feature, results.residual, marker=".", s=72./fig.dpi)
    ax.plot(results.feature, results.ypartial, color='black')
    ax.set_xlabel(feature)
    ax.set_ylabel(f'Residual + {feature} contribution')
    return ax

fig, ax = plt.subplots(figsize=(5, 5))
partialResidualPlot(results, loan_data, 'outcome', 'payment_inc_ratio', fig, ax)
ax.set_xlim(0, 25)
ax.set_ylim(-2.5, 2.5)


plt.tight_layout()
plt.show()

**Figure 5-4:** The black curve shows `payment_inc_ratio`'s effect on log-odds of default; the two residual clouds (defaults above, paid-off below) reflect the binary outcome. Curvature in the line indicates nonlinearity not captured by a linear model.


# Evaluating Classification Models
## Confusion Matrix


The **confusion matrix** cross-tabulates actual vs. predicted outcomes:

| | Predicted + | Predicted − |
|---|---|---|
| **Actual +** | TP | FN |
| **Actual −** | FP | TN |

$$\text{Accuracy} = \frac{TP+TN}{TP+TN+FP+FN}$$


In [ ]:
pred = logit_reg.predict(X)
pred_y = logit_reg.predict(X) == 'default'
true_y = y == 'default'
true_pos = true_y & pred_y
true_neg = ~true_y & ~pred_y
false_pos = ~true_y & pred_y
false_neg = true_y & ~pred_y

conf_mat = pd.DataFrame([[np.sum(true_pos), np.sum(false_neg)], [np.sum(false_pos), np.sum(true_neg)]],
                       index=['Y = default', 'Y = paid off'],
                       columns=['Yhat = default', 'Yhat = paid off'])
print(conf_mat)

Confusion matrix breakdown: 14,337 TP, 14,522 TN, 8,149 FP, 8,334 FN → **Accuracy = 63.65%**. Accuracy treats all errors as equally costly — see precision, recall, and AUC for a fuller picture.


In [ ]:
print(confusion_matrix(y, logit_reg.predict(X)))

Scikit-learn's `confusion_matrix()` returns the same values as a NumPy array (rows = actual, columns = predicted, alphabetical order).


The `dmba` package's `classificationSummary` prints a formatted confusion matrix with accuracy in one line.


In [ ]:
classificationSummary(y, logit_reg.predict(X),
                      class_names=logit_reg.classes_)

63.65% accuracy — substantially better than the 50% naive baseline. However, 37% of actual defaults are missed, which may be costly in practice.


## Precision, Recall, and Specificity
`precision_recall_fscore_support` returns all four metrics in one call.


In [ ]:
conf_mat = confusion_matrix(y, logit_reg.predict(X))
print('Precision', conf_mat[0, 0] / sum(conf_mat[:, 0]))
print('Recall', conf_mat[0, 0] / sum(conf_mat[0, :]))
print('Specificity', conf_mat[1, 1] / sum(conf_mat[1, :]))

- **Precision (0.638):** 64% of predicted defaults are actual defaults.
- **Recall (0.632):** Model catches 63% of all actual defaults.
- **Specificity (0.641):** 64% of non-defaulting loans correctly identified.

All three are similar here because the training data is roughly balanced.


In [ ]:
precision_recall_fscore_support(y, logit_reg.predict(X),
                                labels=['default', 'paid off'])

`precision_recall_fscore_support` returns precision, recall, F1, and support for each class. The **F1-score** is the harmonic mean of precision and recall — it penalizes imbalance between the two.


## ROC Curve
`roc_curve` in scikit-learn computes all threshold-specific FPR/TPR pairs needed to plot an ROC curve.


In [ ]:
fpr, tpr, thresholds = roc_curve(y, logit_reg.predict_proba(X)[:, 0],
                                 pos_label='default')
roc_df = pd.DataFrame({'recall': tpr, 'specificity': 1 - fpr})

ax = roc_df.plot(x='specificity', y='recall', figsize=(4, 4), legend=False)
ax.set_ylim(0, 1)
ax.set_xlim(1, 0)
ax.plot((1, 0), (0, 1))
ax.set_xlabel('specificity')
ax.set_ylabel('recall')


plt.tight_layout()
plt.show()

**Figure 5-6:** The ROC curve plots recall (TPR) vs. specificity (1-FPR) across all cutoffs. Points above the diagonal beat random. The trade-off: lowering the cutoff catches more defaults (↑ recall) but flags more good loans as risky (↓ specificity).


## AUC
`accuracy_score` computes overall accuracy; `roc_auc_score` computes the area under the ROC curve.


In [ ]:
print(np.sum(roc_df.recall[:-1] * np.diff(1 - roc_df.specificity)))
print(roc_auc_score([1 if yi == 'default' else 0 for yi in y], logit_reg.predict_proba(X)[:, 0]))

**AUC = 0.692.** Probabilistic interpretation: a randomly chosen defaulting loan has a 69.2% chance of receiving a higher default probability than a randomly chosen paid-off loan. The two computation methods agree to 7 decimal places.


In [ ]:
fpr, tpr, thresholds = roc_curve(y, logit_reg.predict_proba(X)[:,0],
                                 pos_label='default')
roc_df = pd.DataFrame({'recall': tpr, 'specificity': 1 - fpr})

ax = roc_df.plot(x='specificity', y='recall', figsize=(4, 4), legend=False)
ax.set_ylim(0, 1)
ax.set_xlim(1, 0)
ax.set_xlabel('specificity')
ax.set_ylabel('recall')
ax.fill_between(roc_df.specificity, 0, roc_df.recall, alpha=0.3)


plt.tight_layout()
plt.show()

**Figure 5-7:** Same ROC curve with the AUC region shaded. The shaded area (0.692 of the unit square) represents the model's ranking ability across all possible cutoffs.


# Strategies for Imbalanced Data
## Undersampling


In real-world problems, the class of interest is often rare. Classifiers trained on imbalanced data tend to predict the majority class for almost everything.


In [ ]:
full_train_set = pd.read_csv(FULL_TRAIN_SET_CSV)
print(full_train_set.shape)

Full training set: 119,987 records × 19 columns — larger than `loan_data`, primarily due to additional paid-off loans creating class imbalance.


In [ ]:
print('percentage of loans in default: ',
print(      100 * np.mean(full_train_set.outcome == 'default')))

Only **18.9%** of loans default. A naive "always predict paid-off" model would achieve 81.1% accuracy while catching zero defaults.


In [ ]:
predictors = ['payment_inc_ratio', 'purpose_', 'home_', 'emp_len_',
              'dti', 'revol_bal', 'revol_util']
outcome = 'outcome'
X = pd.get_dummies(full_train_set[predictors], prefix='', prefix_sep='',
                   drop_first=True, dtype=int)
y = full_train_set[outcome]

full_model = LogisticRegression(penalty='l2', C=1e42, solver='liblinear')
full_model.fit(X, y)
print('percentage of loans predicted to default: ',
print(      100 * np.mean(full_model.predict(X) == 'default')))

With imbalanced data, the unweighted model predicts only **0.018%** of loans as defaults — a 1,000× underestimate. The model has been overwhelmed by the majority class.


In [ ]:
(np.mean(full_train_set.outcome == 'default') /
 np.mean(full_model.predict(X) == 'default'))

This ratio (actual / predicted default rate ≫ 1) quantifies the prediction bias from class imbalance.


## Oversampling and Up/Down Weighting


In [ ]:
default_wt = 1 / np.mean(full_train_set.outcome == 'default')
wt = [default_wt if outcome == 'default' else 1 for outcome in full_train_set.outcome]

full_model = LogisticRegression(penalty="l2", C=1e42, solver='liblinear')
full_model.fit(X, y, wt)
print('percentage of loans predicted to default (weighting): ',
print(      100 * np.mean(full_model.predict(X) == 'default')))

**Class weighting** assigns each defaulting loan weight $w = 1/P(\text{default}) \approx 5.3$, making both classes equally influential. After weighting, the model predicts ~60% defaults — an expected overcorrection. Adjust the cutoff threshold accordingly. Scikit-learn's `class_weight='balanced'` automates this.


## Data Generation
`imbalanced-learn` implements SMOTE and related algorithms that determine the number of synthetic cases automatically.


In [ ]:
X_resampled, y_resampled = SMOTE().fit_resample(X, y)
print('percentage of loans in default (SMOTE resampled): ',
      100 * np.mean(y_resampled == 'default'))

full_model = LogisticRegression(penalty="l2", C=1e42, solver='liblinear')
full_model.fit(X_resampled, y_resampled)
print('percentage of loans predicted to default (SMOTE): ',
      100 * np.mean(full_model.predict(X) == 'default'))


X_resampled, y_resampled = ADASYN().fit_resample(X, y)
print('percentage of loans in default (ADASYN resampled): ',
      100 * np.mean(y_resampled == 'default'))

full_model = LogisticRegression(penalty="l2", C=1e42, solver='liblinear')
full_model.fit(X_resampled, y_resampled)
print('percentage of loans predicted to default (ADASYN): ',
print(      100 * np.mean(full_model.predict(X) == 'default')))

**SMOTE** creates synthetic minority records by interpolating between existing ones → balanced at 50% defaults → model predicts ~29% on original data.

**ADASYN** generates more synthetics near the decision boundary → ~49% defaults → predicts ~27%.

Both yield more calibrated predictions than either the unweighted (0.018%) or weighted (60%) approaches.


## Exploring the Predictions


In [ ]:
loan3000 = pd.read_csv(LOAN3000_CSV)

predictors = ['borrower_score', 'payment_inc_ratio']
outcome = 'outcome'

X = loan3000[predictors]
y = loan3000[outcome]

loan_tree = DecisionTreeClassifier(random_state=1, criterion='entropy',
                                   min_impurity_decrease=0.003)
loan_tree.fit(X, y)

loan_lda = LinearDiscriminantAnalysis()
loan_lda.fit(X, y)

logit_reg = LogisticRegression(penalty="l2", solver='liblinear')
logit_reg.fit(X, y)


gam = LinearGAM(s(0) + s(1))
print(gam.gridsearch(X.values, [1 if yi == 'default' else 0 for yi in y]))

Four classifiers are fit to `loan3000` using two predictors, enabling visual comparison of decision boundaries. The GAM uses `gridsearch` to auto-select its smoothing parameter.


In [ ]:
models = {
    'Decision Tree': loan_tree,
    'Linear Discriminant Analysis': loan_lda,
    'Logistic Regression': logit_reg,
    'Generalized Additive Model': gam,
}

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(5, 5))

xvalues = np.arange(0.25, 0.73, 0.005)
yvalues = np.arange(-0.1, 20.1, 0.1)
xx, yy = np.meshgrid(xvalues, yvalues)
X = pd.DataFrame({
    'borrower_score': xx.ravel(),
    'payment_inc_ratio': yy.ravel(),
})

boundary = {}

for n, (title, model) in enumerate(models.items()):
    ax = axes[n // 2, n % 2]
    predict = model.predict(X)
    if 'Generalized' in title:
        Z = np.array([1 if z > 0.5 else 0 for z in predict])
    else:

        Z = np.array([1 if z == 'default' else 0 for z in predict])
    Z = Z.reshape(xx.shape)
    boundary[title] = yvalues[np.argmax(Z > 0, axis=0)]
    boundary[title][Z[-1,:] == 0] = yvalues[-1]

    c = ax.pcolormesh(xx, yy, Z, cmap='Blues', vmin=0.1, vmax=1.3, shading='auto')
    ax.set_title(title)
    ax.grid(True)

plt.tight_layout()
plt.show()

**Figure 5-8:** Decision regions for all four classifiers. Decision Tree → rectangular staircase; LDA & Logistic Regression → nearly identical straight lines; GAM → smooth curve. All agree on the general pattern but differ near the boundary.


In [ ]:
boundary['borrower_score'] = xvalues
boundaries = pd.DataFrame(boundary)

fig, ax = plt.subplots(figsize=(5, 4))
boundaries.plot(x='borrower_score', ax=ax)
ax.set_ylabel('payment_inc_ratio')
ax.set_ylim(0, 20)


plt.tight_layout()
plt.show()